# Determination of the distance to Pleiades using TOPCAT

This tutorial has been adapted from the [TOPCAT tutorial](https://zenodo.org/records/10720670) developed by the Spanish Virtual Observatory. We will use data from the [Tycho-Gaia Astrometric Solution (TGAS)](https://cdsarc.cds.unistra.fr/viz-bin/cat/I/337)  catalogue to determine the mean parallax of the stars in the Pleiades open star cluster, thus obtaining its distance. For this, we will access photometric catalogues, cross-match tables, filter sources, create subsets, and represent the results using different kind of plots.

## Science case

Stars do not form isolated in space. They are born in large groups from the same natal interstellar cloud. We call them **star clusters**. Because all the stars in a given open cluster have the same origin,
they share important properties, having the same age, similar chemistry, and the same kinematics.

The Pleiades (M45) is one of the star clusters nearest to Earth. It is visible by naked eye from both
hemispheres and contains thousands of stars.

## Workflow

| Block | What we do |
|:--|:--|
| **Block 0** | Setup: install and import the packages |
| **Block 1** | Data discovery: download TGAS data around the Pleiades |
| **Block 2** | Select comoving sources using proper motions |
| **Block 3** | Identify the Pleiades members using parallaxes |
| **Block 4** | Calculate the distance to the Pleiades |
| **Bonus** | 2MASS photometry, comparison with another study, and sky view with Aladin |

**How to use this notebook:** run the cells one by one, from top to bottom, with `Shift + Enter`.

---
## Block 0: Setup

First, we install all the packages we will need. (If you already have them installed, you can skip this cell.)

In [4]:
!pip install -r requirements_topcat.txt

Now we import them:

In [5]:
# Standard Library
from pathlib import Path

# Astronomy tools
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.table import Table

# Access astronomical databases
import pyvo
from astroquery.simbad import Simbad
from astroquery.vizier import Vizier
from astroquery.xmatch import XMatch
from astroquery.vo_conesearch import conesearch
from astroquery.mast import Catalogs

# Sky visualization
from ipyaladin import Aladin

#Other tools
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
import pandas as pd
import plotly.express as px
import seaborn as sns

Finally, we choose a common style for all the plots in this notebook:

In [ ]:
# Common style for all the plots (white background with a light grid)
sns.set_theme(style="whitegrid", context="notebook")

---
## Block 1: Data discovery

Let's begin by acquiring TGAS data in the Pleiades region using astroquery. We ask for all the sources within 5 degrees of the cluster centre.

In [6]:
# Query the TGAS catalogue around the Pleiades (the radius is given in degrees)
catalog_data = Catalogs.query_object(catalog="TGAS", objectname='Pleiades', radius=5)

# Round the columns we are going to use, so the tables are easier to read
catalog_data['ra'] = catalog_data['ra'].round(6)
catalog_data['dec'] = catalog_data['dec'].round(6)
catalog_data['pmra'] = catalog_data['pmra'].round(2)
catalog_data['pmdec'] = catalog_data['pmdec'].round(2)
catalog_data['parallax'] = catalog_data['parallax'].round(2)
catalog_data['parallax_error'] = catalog_data['parallax_error'].round(2)
catalog_data['phot_g_mean_mag'] = catalog_data['phot_g_mean_mag'].round(2)

# Show the most relevant columns
print(catalog_data[['hip', 'tycho2_id', 'ra', 'dec', 'parallax', 'parallax_error','pmra','pmdec','phot_g_mean_mag']])

 hip   tycho2_id      ra       dec    ...  pmra pmdec  phot_g_mean_mag
----- ----------- --------- --------- ... ----- ------ ---------------
   -- 1800-1974-1 56.557353 24.196562 ... 17.72 -12.33            7.24
   -- 1800-1908-1 56.613754 24.254813 ... 19.96 -44.29            7.36
   --   1799-79-1 56.465239 24.038689 ... 19.72 -44.88            7.84
   -- 1800-1579-1 56.767652 23.995026 ... 22.72 -44.93            8.31
   -- 1800-1607-1 56.830695 24.138932 ... 17.88 -43.46            8.24
   -- 1800-2201-1 56.837754  24.11608 ... 20.89 -44.68            6.33
   --   1799-26-1 56.359021 24.034958 ... 21.31 -46.95            8.01
   -- 1800-1935-1 56.791982 24.276466 ... 20.43 -43.95            9.33
17704          -- 56.872808 24.288143 ... 18.48 -46.91            6.83
   -- 1800-1567-1 56.851815  23.91449 ... 18.85 -43.76            7.31
  ...         ...       ...       ... ...   ...    ...             ...
   --  1257-197-1 58.786139 19.554936 ... 52.16  -0.24             9.6
   -- 

Let's visualize the sources we have obtained in a **proper motion diagram**. Note the overdensity around (pmRA, pmDec) = (20, -45) mas/yr: those stars move together in the sky.

In [ ]:
# Create a new figure with a size of 10 inches by 7 inches
plt.figure(figsize=(10, 7))

# Plot a scatter plot for the 'catalog_data' dataset (all sources) with blue color
plt.scatter(catalog_data['pmra'], catalog_data['pmdec'], s=10, c='steelblue', label="All Sources", alpha=0.6, edgecolors='w', linewidth=0.5)
# Plot 'pmra' and 'pmdec' for all sources, with small blue points, some transparency, and white edges

# Set the limits for the x-axis (Proper Motion RA)
plt.xlim(-25, 55)  # Set the x-axis range from -25 to 55 for proper motion in RA

# Set the limits for the y-axis (Proper Motion Dec)
plt.ylim(-55, 15)  # Set the y-axis range from -55 to 15 for proper motion in Dec

# Set the label for the x-axis
plt.xlabel("Proper Motion RA (mas/yr)", fontsize=12, fontweight='bold')  # Label the x-axis as 'Proper Motion RA (mas/yr)'

# Set the label for the y-axis
plt.ylabel("Proper Motion Dec (mas/yr)", fontsize=12, fontweight='bold')  # Label the y-axis as 'Proper Motion Dec (mas/yr)'

# Set the title for the plot
plt.title("Proper Motion in RA and Dec", fontsize=14, fontweight='bold')  # Set the title of the plot

# Add a legend in the upper-right corner, without a frame
plt.legend(frameon=False, loc='upper right', fontsize=10)  # Display the legend at the upper-right corner, without a background frame

# Adjust the appearance of the ticks on both axes
plt.tick_params(axis='both', which='major', labelsize=10)  # Set the size of the tick labels on both axes

# Add a grid to the plot for better readability of data
plt.grid(True, linestyle='--', color='gray', alpha=0.5)  # Display a grid with dashed lines and semi-transparent gray color

# Show the plot on the screen
plt.show()  # Display the plot with all the settings

---
## Block 2: Select comoving sources

We will select Pleiades candidate members applying some filtering criteria in PMRA and PMDEC (in the original tutorial the selection of the region is done by hand):

* 15 < pmRA < 25 mas/yr
* -50 < pmDec < -40 mas/yr

In [8]:
# Select comoving sources
position_mask = (
    (catalog_data['pmra'] > 15) &
    (catalog_data['pmra'] < 25) &
    (catalog_data['pmdec'] > -50) &
    (catalog_data['pmdec'] < -40)
)

comoving = catalog_data[position_mask]

print(comoving[['hip', 'tycho2_id', 'ra', 'dec', 'parallax', 'parallax_error','pmra','pmdec','phot_g_mean_mag']])

 hip   tycho2_id      ra       dec    ...  pmra pmdec  phot_g_mean_mag
----- ----------- --------- --------- ... ----- ------ ---------------
   -- 1800-1908-1 56.613754 24.254813 ... 19.96 -44.29            7.36
   --   1799-79-1 56.465239 24.038689 ... 19.72 -44.88            7.84
   -- 1800-1579-1 56.767652 23.995026 ... 22.72 -44.93            8.31
   -- 1800-1607-1 56.830695 24.138932 ... 17.88 -43.46            8.24
   -- 1800-2201-1 56.837754  24.11608 ... 20.89 -44.68            6.33
   --   1799-26-1 56.359021 24.034958 ... 21.31 -46.95            8.01
   -- 1800-1935-1 56.791982 24.276466 ... 20.43 -43.95            9.33
17704          -- 56.872808 24.288143 ... 18.48 -46.91            6.83
   -- 1800-1567-1 56.851815  23.91449 ... 18.85 -43.76            7.31
   -- 1800-1620-1 56.657091 23.787523 ... 19.65 -47.11            10.0
  ...         ...       ...       ... ...   ...    ...             ...
16431          -- 52.890208  26.26529 ... 20.81 -42.12           10.51
17325 

Now, the found candidates are overplotted in the previous proper motion diagram. The dashed box shows the region we have selected.

In [ ]:
# Create a new figure with a size of 10 inches by 7 inches
plt.figure(figsize=(10, 7))

# Plot a scatter plot for the 'catalog_data' dataset (all sources) with blue color
plt.scatter(catalog_data['pmra'], catalog_data['pmdec'], s=10, c='steelblue', label="All Sources", alpha=0.6, edgecolors='w', linewidth=0.5)
# Plot 'pmra' and 'pmdec' for all sources, with small blue points, some transparency, and white edges

# Plot a scatter plot for the 'comoving' dataset (comoving sources) with red color
plt.scatter(comoving['pmra'], comoving['pmdec'], s=30, c='red', label="Comoving Sources", alpha=0.8, edgecolors='black', linewidth=0.5)
# Plot 'pmra' and 'pmdec' for comoving sources, larger red points, black edges, and more solid color (less transparency)

# Draw the selection box (same limits we used in the previous cell)
plt.plot([15, 25, 25, 15, 15], [-50, -50, -40, -40, -50], color='black', linestyle='--', linewidth=1.5, label="Selection box")

# Set the limits for the x-axis (Proper Motion RA)
plt.xlim(-25, 55)  # Set the x-axis range from -25 to 55 for proper motion in RA

# Set the limits for the y-axis (Proper Motion Dec)
plt.ylim(-55, 15)  # Set the y-axis range from -55 to 15 for proper motion in Dec

# Set the label for the x-axis
plt.xlabel("Proper Motion RA (mas/yr)", fontsize=12, fontweight='bold')  # Label the x-axis as 'Proper Motion RA (mas/yr)'

# Set the label for the y-axis
plt.ylabel("Proper Motion Dec (mas/yr)", fontsize=12, fontweight='bold')  # Label the y-axis as 'Proper Motion Dec (mas/yr)'

# Set the title for the plot
plt.title("Proper Motion in RA and Dec", fontsize=14, fontweight='bold')  # Set the title of the plot

# Add a legend in the upper-right corner, without a frame
plt.legend(frameon=False, loc='upper right', fontsize=10)  # Display the legend at the upper-right corner, without a background frame

# Adjust the appearance of the ticks on both axes
plt.tick_params(axis='both', which='major', labelsize=10)  # Set the size of the tick labels on both axes

# Add a grid to the plot for better readability of data
plt.grid(True, linestyle='--', color='gray', alpha=0.5)  # Display a grid with dashed lines and semi-transparent gray color

# Show the plot on the screen
plt.show()  # Display the plot with all the settings

---
## Block 3: Identification of Pleiades members

We will now use the parallax to refine the selection and identify the Pleiades members. Let's look at the parallax distribution of the comoving sources:

In [ ]:
# Create a new figure with a size of 10 inches by 6 inches
plt.figure(figsize=(10, 6))

# Plot histogram of parallax values
plt.hist(comoving['parallax'], bins=30, edgecolor='black', color='cornflowerblue')

# Labels and title
plt.xlabel("Parallax (mas)", fontsize=12, fontweight='bold')
plt.ylabel("Number of Stars", fontsize=12, fontweight='bold')
plt.title("Parallax Distribution: Comoving Sources", fontsize=14, fontweight='bold')

# Show the plot
plt.show()

There are some outliers, most likely not cluster members. We want to exclude those parallax
outliers, and limit our sources to parallaxes between 6
and 9. We will name this new list of sources as '*cluster*'.

In [11]:
# Select stars in the cluster
cluster_mask = (
    (comoving['parallax'] > 6) &
    (comoving['parallax'] < 9)
)

# Extract cluster sources
cluster = comoving[cluster_mask]

print(cluster[['hip', 'tycho2_id', 'ra', 'dec', 'parallax', 'parallax_error','pmra','pmdec','phot_g_mean_mag']])

 hip   tycho2_id      ra       dec    ...  pmra pmdec  phot_g_mean_mag
----- ----------- --------- --------- ... ----- ------ ---------------
   -- 1800-1908-1 56.613754 24.254813 ... 19.96 -44.29            7.36
   --   1799-79-1 56.465239 24.038689 ... 19.72 -44.88            7.84
   -- 1800-1579-1 56.767652 23.995026 ... 22.72 -44.93            8.31
   -- 1800-1607-1 56.830695 24.138932 ... 17.88 -43.46            8.24
   -- 1800-2201-1 56.837754  24.11608 ... 20.89 -44.68            6.33
   --   1799-26-1 56.359021 24.034958 ... 21.31 -46.95            8.01
   -- 1800-1935-1 56.791982 24.276466 ... 20.43 -43.95            9.33
17704          -- 56.872808 24.288143 ... 18.48 -46.91            6.83
   -- 1800-1567-1 56.851815  23.91449 ... 18.85 -43.76            7.31
   -- 1800-1620-1 56.657091 23.787523 ... 19.65 -47.11            10.0
  ...         ...       ...       ... ...   ...    ...             ...
   --   1261-24-1 59.590162 21.257481 ... 17.83 -45.13           11.18
   -- 

Let's see how the plot looks now. The dashed lines mark the parallax limits we have used:

In [ ]:
# Determine the minimum and maximum values of parallax from both 'comoving' and 'cluster' datasets
min_value = min(comoving['parallax'].min(), cluster['parallax'].min())  # Minimum parallax value from both datasets
max_value = max(comoving['parallax'].max(), cluster['parallax'].max())  # Maximum parallax value from both datasets

# Define the number of bins for the histogram
bins = np.linspace(min_value, max_value, 30)  # Create an array of bins between the min and max parallax values

# Create a figure with a specified size
plt.figure(figsize=(10, 6))  # Set the figure size to 10x6 inches

# Plot the first histogram for the 'comoving' dataset
plt.hist(comoving['parallax'], bins=bins, edgecolor='black', color='cornflowerblue', alpha=0.6, label="Comoving Sources")
# Plot 'parallax' data from the 'comoving' dataset with blue color and transparency of 0.6

# Plot the second histogram for the 'cluster' dataset
plt.hist(cluster['parallax'], bins=bins, edgecolor='black', color='salmon', alpha=0.6, label="Cluster")
# Plot 'parallax' data from the 'cluster' dataset with red color and transparency of 0.6

# Mark the parallax limits of our selection (6 and 9 mas)
plt.axvline(6, color='black', linestyle='--', linewidth=1.5)
plt.axvline(9, color='black', linestyle='--', linewidth=1.5)

# Set labels for the x and y axes
plt.xlabel("Parallax (mas)", fontsize=12, fontweight='bold')  # Set x-axis label as 'Parallax (mas)'
plt.ylabel("Number of Stars", fontsize=12, fontweight='bold')  # Set y-axis label as 'Number of Stars'

# Set the title for the histogram
plt.title("Parallax Distribution: Comoving vs Cluster", fontsize=14, fontweight='bold')  # Title of the histogram

# Add a legend to distinguish between the two datasets
plt.legend(frameon=False, loc='upper right', fontsize=10)  # Display the legend in the upper-right corner

# Customize tick parameters (size of tick labels)
plt.tick_params(axis='both', which='major', labelsize=10)  # Set the size of the tick labels on both axes

# Display the plot
plt.show()  # Show the histogram

---
## Block 4: Calculating the distance to Pleiades

In order to calculate realistic distances to the Pleiades, we need accurate parallaxes. Distance in parsec is the reciprocal of parallax
in arcsec:

$$d\,[\mathrm{pc}] = \frac{1}{\varpi\,[\mathrm{arcsec}]} = \frac{1000}{\varpi\,[\mathrm{mas}]}$$

However, inverting parallaxes to get distances is problematic if parallax errors are large, let's say, larger than 10%. Thus, in the next step, we will keep only sources with good parallaxes.

In [13]:
# Select stars with good distance (parallax error smaller than 10%)
distance_mask = (
    (cluster['parallax_error']/cluster['parallax'] < 0.1)
)

good_distance = cluster[distance_mask]

# Compute the distance in parsec (the parallax is in mas)
good_distance['distance'] = 1000 / good_distance['parallax']
good_distance['distance'] = good_distance['distance'].round(2)

print(good_distance[['hip', 'tycho2_id', 'ra', 'dec', 'parallax', 'parallax_error','pmra','pmdec','phot_g_mean_mag','distance']])

# Optional: uncomment the following lines to save the table as a CSV file
# Convert the data to a pandas DataFrame
#df = good_distance.to_pandas()
# Save the DataFrame as a CSV file
#df.to_csv('good_distance.csv', index=False)

 hip   tycho2_id      ra       dec    ...  pmra pmdec  phot_g_mean_mag distance
----- ----------- --------- --------- ... ----- ------ --------------- --------
   -- 1800-1908-1 56.613754 24.254813 ... 19.96 -44.29            7.36    129.2
   --   1799-79-1 56.465239 24.038689 ... 19.72 -44.88            7.84   134.95
   -- 1800-1579-1 56.767652 23.995026 ... 22.72 -44.93            8.31   140.25
   -- 1800-1607-1 56.830695 24.138932 ... 17.88 -43.46            8.24   130.89
   -- 1800-2201-1 56.837754  24.11608 ... 20.89 -44.68            6.33    126.9
   --   1799-26-1 56.359021 24.034958 ... 21.31 -46.95            8.01   121.51
   -- 1800-1935-1 56.791982 24.276466 ... 20.43 -43.95            9.33   139.86
17704          -- 56.872808 24.288143 ... 18.48 -46.91            6.83    132.1
   -- 1800-1567-1 56.851815  23.91449 ... 18.85 -43.76            7.31   130.72
   -- 1800-1620-1 56.657091 23.787523 ... 19.65 -47.11            10.0    129.7
  ...         ...       ...       ... ..

Now we plot the results, together with a Gaussian fit to the good parallaxes:

In [ ]:
# Get the data limits for the three datasets
min_value = min(comoving['parallax'].min(), cluster['parallax'].min(), good_distance['parallax'].min())
max_value = max(comoving['parallax'].max(), cluster['parallax'].max(), good_distance['parallax'].max())

# Define the number of bins you want (e.g., 30 bins)
bins = np.linspace(min_value, max_value, 30)

# Figure size
plt.figure(figsize=(10, 6))

# First histogram (Comoving)
plt.hist(comoving['parallax'], bins=bins, edgecolor='black', color='cornflowerblue', alpha=0.6, label="Comoving Sources")

# Second histogram (Cluster)
plt.hist(cluster['parallax'], bins=bins, edgecolor='black', color='salmon', alpha=0.6, label="Cluster")

# Third histogram (Good Distance)
plt.hist(good_distance['parallax'], bins=bins, edgecolor='black', color='seagreen', alpha=0.6, label="Good Distance Sources")

# Gaussian fit to the 'parallax' data from good_distance
mu, std = norm.fit(good_distance['parallax'])  # Fit the normal distribution

# Generate the normal distribution curve
xmin, xmax = plt.xlim()  # Get the limits of the X-axis
x = np.linspace(xmin, xmax, 100)  # Create a range of values for the curve
p = norm.pdf(x, mu, std)  # Normal probability density function (PDF)

# Adjust the height of the curve to match the histogram
# The height of the Gaussian curve should be scaled by the number of elements in the data and the bin width
bin_width = bins[1] - bins[0]  # The width of each bin
n_total = len(good_distance['parallax'])  # Total number of elements in good_distance['parallax']
p_scaled = p * n_total * bin_width  # Scale the curve

# Choose a color for the curve that stands out over the histograms
gaussian_color = 'black'

# Overlay the Gaussian curve on the Good Distance histogram
plt.plot(x, p_scaled, color=gaussian_color, linewidth=2, label=f'Gaussian Fit: μ = {mu:.2f}, σ = {std:.2f}')

# Labels and title
plt.xlabel("Parallax (mas)", fontsize=12, fontweight='bold')
plt.ylabel("Number of Stars", fontsize=12, fontweight='bold')
plt.title("Parallax Distribution: Comoving vs Cluster vs Good Distance", fontsize=14, fontweight='bold')

# Legend
plt.legend(frameon=False, loc='upper right', fontsize=10)

# Axis style
plt.tick_params(axis='both', which='major', labelsize=10)

# Show the figure
plt.show()

How far away is the Pleiades open cluster? Let's calculate the mean distance using sources with good parallaxes.

In [15]:
# Compute mean and standard deviation

mean_distance = good_distance['distance'].mean()
std_distance = good_distance['distance'].std()

print("="*50)
print(f"\033[1;32mDistance (mean): {mean_distance:.2f} pc\033[0m")
print(f"\033[1;31mDistance (standard deviation): {std_distance:.2f} pc\033[0m")

print("="*50)

Distance (mean): 134.45 pc
Distance (standard deviation): 6.85 pc


Finally, we will visualize the sources we have identified as Pleiades members in an interactive cube plot:

In [ ]:
# Create a DataFrame with the necessary data from 'good_distance'
data = good_distance[['ra', 'dec', 'distance']]

# Create the interactive 3D plot
fig = px.scatter_3d(
    data,
    x='ra',
    y='dec',
    z='distance',
    color='distance',  # Use distance to color the points
    labels={'ra': 'RA (deg)', 'dec': 'Dec (deg)', 'distance': 'Distance (pc)'},  # Axis labels
    color_continuous_scale='Viridis',  # Attractive color palette
    title="Pleiades members: RA, Dec and Distance",  # Title of the plot
)

# Additional styling to make the plot more aesthetically pleasing
fig.update_traces(marker=dict(size=5, opacity=0.8, line=dict(width=0.5, color='DarkSlateGrey')))  # Customize the points
fig.update_layout(
    scene=dict(
        xaxis_title='RA (deg)',  # X-axis title
        yaxis_title='Dec (deg)',  # Y-axis title
        zaxis_title='Distance (pc)',  # Z-axis title
    ),
    margin=dict(r=10, b=10, l=10, t=40),  # Margins for a clean presentation
    template='plotly_dark'  # Dark background style
)

# Display the plot
fig.show()

---
## Bonus

### Bonus 1: How many stars have 2MASS photometry?

In order to answer this question, we can query the 2MASS-PSC catalog and then perform a cross-match with the sources we have obtained:

In [17]:
#Search 2MASS-PSC catalogs

Vizier.find_catalogs("2MASS-PSC")

OrderedDict([('II/246', </>), ('II/281', </>)])

The 2MASS-PSC catalogue is `II/246`. Let's obtain the coordinates of the sources in `good_distance` and then query the 2MASS-PSC catalog around each one of those coordinates (this cell can take a few minutes, as it sends one query per source).

In [18]:
# Define search radius
search_radius = 5 * u.arcsec

# Initialize Vizier (row_limit=-1 means no limit in the number of rows of the output)
viz = Vizier(columns=["*"], row_limit=-1)

all_matches = []

# Coordinates of our sources
source_coords = SkyCoord(good_distance["ra"], good_distance["dec"], unit=(u.deg, u.deg))

# Query 2MASS around each source
for i, coord in enumerate(source_coords):

    result = viz.query_region(coord, radius=search_radius, catalog="II/246")

    if result:

        table = result[0]

        # Angular distance between our source and the 2MASS sources found
        matched_coords = SkyCoord(ra=table['RAJ2000'],dec=table['DEJ2000'],unit=(u.deg, u.deg),frame='icrs')
        separations = coord.separation(matched_coords).to(u.arcsec)

        table['query_source_idx'] = i
        table['ang_dist'] = separations

        all_matches.append(table.to_pandas())

# Combine all results into one table, keeping only the closest 2MASS source for each of our sources
matched_results = pd.concat(all_matches, ignore_index=True).sort_values(by='ang_dist',ascending=True).drop_duplicates(subset='query_source_idx',keep='first').reset_index(drop=True)
matched_results

,RAJ2000,DEJ2000,2MASS,Jmag,e_Jmag,Hmag,e_Hmag,Kmag,e_Kmag,Qflg,Rflg,Bflg,Cflg,Xflg,Aflg,query_source_idx,ang_dist
0,58.589973,24.075645,03542159+2404323,8.687,0.021,8.480,0.034,8.424,0.021,AAA,112,111,000,0,0,96,0.555821
1,56.767553,23.995174,03470421+2359426,7.770,0.018,7.658,0.051,7.642,0.023,AAA,111,111,000,0,0,2,0.624413
2,56.001108,24.556950,03440026+2433250,7.633,0.037,7.618,0.044,7.583,0.018,AAA,111,111,000,0,0,27,0.625691
3,55.961274,24.247465,03435070+2414508,9.833,0.024,9.429,0.036,9.344,0.018,AAA,222,111,000,0,0,21,0.641453
4,56.014746,24.504206,03440353+2430151,9.405,0.023,9.118,0.029,8.953,0.020,AAA,222,111,000,0,0,24,0.645034
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143,57.485860,24.349070,03495660+2420566,7.346,0.020,7.367,0.016,7.350,0.017,AAA,111,111,000,0,0,39,0.957246
144,56.950462,23.218140,03474811+2313053,9.760,0.020,9.436,0.022,9.324,0.017,AAA,222,111,000,0,0,49,0.957958
145,52.868053,21.821951,03312833+2149190,8.071,0.023,7.890,0.017,7.824,0.023,AAA,111,111,000,0,0,142,0.980278
146,57.163840,21.925020,03483932+2155300,7.557,0.027,7.400,0.023,7.320,0.021,AAA,111,111,000,0,0,108,0.997948


In [ ]:
print("Sources in our list:", len(good_distance))
print("Sources with 2MASS photometry:", len(matched_results))

### Bonus 2: Compare with another study

We can also compare our resulting member list with another census of Pleiades members to see how many objects they have in common. As an example, we will cross-match our table with a catalogue from VizieR.

Let's select our catalog by searching it by keyword:

In [19]:
Vizier.find_catalogs("Pleiades")

OrderedDict([('I/90', </>),
             ('I/163', </>),
             ('I/258', </>),
             ('II/131', </>),
             ('J/ApJ/348/557', </>),
             ('J/ApJ/602/L117', </>),
             ('J/ApJ/822/81', </>),
             ('J/ApJ/901/91', </>),
             ('J/ApJ/904/140', </>),
             ('J/ApJ/916/66', </>),
             ('J/ApJ/921/117', </>),
             ('J/ApJ/923/23', </>),
             ('J/ApJ/926/132', </>),
             ('J/ApJS/85/315', </>),
             ('J/ApJS/91/625', </>),
             ('J/ApJS/102/75', </>),
             ('J/ApJS/172/663', </>),
             ('J/A+A/299/696', </>),
             ('J/A+A/320/74', </>),
             ('J/A+A/323/139', </>),
             ('J/A+A/329/101', </>),
             ('J/A+A/332/575', </>),
             ('J/A+A/335/183', </>),
             ('J/A+A/341/751', </>),
             ('J/A+A/400/891', </>),
             ('J/A+A/410/671', </>),
             ('J/A+A/416/125', </>),
             ('J/A+A/463/579', </>),

Select catalog `J/ApJS/172/663` (Stauffer+ 2007).

In [20]:
Vizier.get_catalogs("J/ApJS/172/663")

TableList with 4 tables:
	'0:J/ApJS/172/663/table2' with 15 column(s) and 50 row(s) 
	'1:J/ApJS/172/663/table5' with 11 column(s) and 50 row(s) 
	'2:J/ApJS/172/663/table4' with 8 column(s) and 50 row(s) 
	'3:J/ApJS/172/663/notes' with 2 column(s) and 34 row(s) 

We are interested in Table 2, so let's select only that table:

In [21]:
#We select to have no limit in our output by setting it to -1:
Vizier.ROW_LIMIT = -1
#The [0] at the end of the line selects only the first element in the table list, which is table 2 from the catalog as we have seen
catalog_data2=Vizier.get_catalogs("J/ApJS/172/663/table2")[0]
print(catalog_data2)

 RAJ2000    DEJ2000    Jmag  q_Jmag ...  Vmag Icmag Ref        SimbadName       
   deg        deg      mag          ...  mag   mag                              
---------- ---------- ------ ------ ... ----- ----- --- ------------------------
 51.898273  24.528660 10.781      A ...    -- 11.85  22    Cl* Melotte 22 DH 001
 51.925262  23.803688  9.066      A ... 10.30    --   4  Cl* Melotte 22 PELS 121
 51.976067  24.936478 12.880      A ...    -- 14.34  22    Cl* Melotte 22 DH 003
 52.006481  23.078499 13.525      A ...    -- 15.05  22    Cl* Melotte 22 DH 004
 52.168613  25.607782 10.198      A ... 11.75    --   9 Cl* Melotte 22 AK III-59
 52.200249  25.575842 13.072      A ...    -- 14.75  22    Cl* Melotte 22 DH 006
 52.203186  26.499350 10.429      A ...    -- 11.16  22    Cl* Melotte 22 DH 007
 52.355843  25.652304  8.459      A ...  9.43    --   9 Cl* Melotte 22 AK III-79
 52.409874  24.510546 10.318      A ...    -- 11.19  22    Cl* Melotte 22 DH 008
       ...        ...    ...

Let's now perform a pair match, setting the max distance to 2.0 arcsec this time.

In [ ]:
# First, we convert to SkyCoord objects. Lets call "gd" to those of the "good distance" sources, and "c2" to those of "catalog data 2"
coords_gd = SkyCoord(good_distance["ra"], good_distance["dec"], unit=(u.deg, u.deg))
coords_c2 = SkyCoord(catalog_data2["RAJ2000"], catalog_data2["DEJ2000"], unit=(u.deg, u.deg))

# Define matching radius
radius = 2 * u.arcsec

# Perform the pair match by using search_around_sky with the coordinates we have obtained,
# in return we obtain the index in each table that corresponds to the matching source
# in the other table, as well as the separations on-sky and 3D, which we set in blank as we won't need them.
# Note the order of the output: first the indices in the catalogue we search in (c2), then the indices in our table (gd)
idx_c2, idx_gd, _, _ = coords_gd.search_around_sky(coords_c2, radius)

How many objects are confirmed as kinematical members of the Pleiades?

In [ ]:
print("Sources in our list:", len(good_distance))
print("Sources in common with Stauffer+ 2007:", len(idx_c2))

These are the sources in common, as they appear in the Stauffer+ 2007 catalogue:

In [23]:
catalog_data2[idx_c2]

RAJ2000,DEJ2000,Jmag,q_Jmag,Hmag,q_Hmag,Ksmag,q_Ksmag,r_Ksmag,2M,Bmag,Vmag,Icmag,Ref,SimbadName
deg,deg,mag,,mag,,mag,,,,mag,mag,mag,,
float64,float64,float32,str1,float32,str1,float32,str1,uint8,str2,float32,float32,float32,int16,str25
52.409874,24.510546,10.318,A,9.856,A,9.698,A,1,2M,--,--,11.19,22,Cl* Melotte 22 DH 008
52.890076,26.265507,9.514,A,9.222,A,9.068,A,1,2M,11.45,10.77,--,4,Cl* Melotte 22 PELS 008
53.274277,22.134232,9.897,A,9.522,A,9.425,A,1,2M,12.25,11.43,--,4,Cl* Melotte 22 PELS 004
53.307938,23.006475,10.259,A,9.787,A,9.669,A,1,2M,12.93,11.98,--,4,Cl* Melotte 22 PELS 123
53.507523,24.880957,9.582,A,9.269,A,9.178,A,1,2M,11.58,10.88,--,4,Cl* Melotte 22 PELS 005
53.530495,24.344501,8.553,A,8.312,A,8.274,A,1,2M,10.08,9.57,--,4,Cl* Melotte 22 PELS 006
53.882030,22.823627,8.831,A,8.599,A,8.541,A,1,2M,10.40,9.86,--,4,Cl* Melotte 22 PELS 124
54.350204,22.350985,8.141,A,7.946,A,7.902,A,1,2M,9.71,9.20,--,4,Cl* Melotte 22 PELS 003


In the following histogram we can see how our sources are among the brightest sources in the Stauffer+ 2007 catalogue. This is because TGAS is a shallow survey with a limiting magnitude around B=13.

In [ ]:
# Use the same bins for both histograms, so they can be compared
bins = np.linspace(catalog_data2['Bmag'].min(), catalog_data2['Bmag'].max(), 30)

# Create the figure and axis
fig, ax = plt.subplots(figsize=(10, 6))

# Histogram 1: full catalog
ax.hist(
    catalog_data2['Bmag'],
    bins=bins,
    color='cornflowerblue',
    alpha=0.6,
    edgecolor='black',
    label="Full catalog (Stauffer+ 2007)"
)

# Histogram 2: cross-matched subset
ax.hist(
    catalog_data2[idx_c2]['Bmag'],
    bins=bins,
    color='salmon',
    alpha=0.8,
    edgecolor='black',
    label="Sources in common with our list"
)

# Axis titles
ax.set_title("Distribution of B-band Magnitudes", fontsize=14, fontweight='bold')
ax.set_xlabel("B (mag)", fontsize=12, fontweight='bold')
ax.set_ylabel("Number of Stars", fontsize=12, fontweight='bold')

# Legend
ax.legend(frameon=False, loc='upper left', fontsize=10)

# Tick styling
ax.tick_params(axis='both', which='major', labelsize=10)

# Show the plot
plt.show()

### Bonus 3: What do these stars look like?

Let's use now Aladin to visualize the position in the sky of our 148 sources (table `good_distance`).

In [30]:
#Load a sky map in Aladin:
aladin1 = Aladin(
    survey="Collection/Images/Optical/DSS colored",
    fov=9,
    target="Pleiades"
)
aladin1


Aladin(survey='Collection/Images/Optical/DSS colored')

Once the sky map is displayed, we overplot our sources on it (go back to the map above to see them):

In [31]:
aladin1.add_table(good_distance, name="Pleiades sources")

---
## Even more

Many other functionalities are available in TOPCAT: concatenate tables, cross-match multiple tables,
save tables in LaTeX format, and many more…. For further information in these and many more functionalities, we refer the user to the TOPCAT manual web page:

http://www.star.bristol.ac.uk/~mbt/topcat/